### [Bước 1] Kiểm tra cấu hình GPU được cấp phát


In [ ]:
!nvidia-smi


Sat Nov 14 10:39:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### [Bước 2] Kết nối Google Drive


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

raw_dataset = '/content/drive/MyDrive/raw_fire_dataset'
zip_dataset = '/content/drive/MyDrive/fire_clean.zip'

print(f"Kiểm tra Google Drive:")
print(f"- Thư mục dataset gốc (/MyDrive/raw_fire_dataset): {'Đã tìm thấy' if os.path.exists(raw_dataset) else 'Chưa có'}")
print(f"- File nén sạch (/MyDrive/fire_clean.zip): {'Đã có sẵn' if os.path.exists(zip_dataset) else 'Chưa tạo'}")

Mounted at /content/drive
Kiểm tra Google Drive:
- Thư mục dataset gốc (/MyDrive/raw_fire_dataset): Đã tìm thấy
- File nén sạch (/MyDrive/fire_clean.zip): Chưa tạo


### [Bước 3] Tải mã nguồn YOLOv5 & Cài đặt môi trường


In [ ]:
%cd /content
!rm -rf yolov5
!git clone https://github.com/ultralytics/yolov5.git
%cd /content/yolov5

# Cài đặt các thư viện cần thiết
%pip install -qr requirements.txt
%pip install -q onnx onnxruntime streamlit pandas matplotlib seaborn

# Tắt bảng hỏi wandb 30 giây
%env WANDB_MODE=disabled

# Vá lỗi tương thích NumPy mới trong utils/plots.py của YOLOv5
!python -c "import re; p='utils/plots.py'; c=open(p, encoding='utf-8').read(); c=c.replace('annotator.im.save(fname)', 'cv2.imwrite(str(fname), np.asarray(annotator.im)[:, :, ::-1]) if not hasattr(annotator.im, \"save\") else annotator.im.save(fname)'); open(p, 'w', encoding='utf-8').write(c)"

import torch
from IPython.display import Image, display
print(f"Setup hoàn tất. PyTorch {torch.__version__} - GPU sẵn sàng: {torch.cuda.is_available()}")


/content
Cloning into 'yolov5'...
remote: Enumerating objects: 18881, done.
remote: Counting objects: 100% (410/410), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 18881 (delta 352), reused 336 (delta 336), pack-reused 18471 (from 3)
Receiving objects: 100% (18881/18881), 17.70 MiB | 26.97 MiB/s, done.
Resolving deltas: 100% (12886/12886), done.
/content/yolov5
env: WANDB_MODE=disabled
Setup hoàn tất. PyTorch 2.14.1+cu130 - GPU sẵn sàng: True


### [Bước 4] TIỀN XỬ LÝ ẢNH THÔNG MINH (Smart Cache Pipeline)
* **Nếu Drive đã có `fire_clean.zip`:** Giải nén ngay trong và Train ngay lập tức
* **Nếu chưa có:** Tiến hành quét sạch ảnh hỏng, lọc ảnh nhỏ, chuẩn hóa Bounding Box, thêm negative samples từ `MyDrive/fire` Nén và lưu thẳng `fire_clean.zip` lên Drive để các lần sau dùng.


In [ ]:
import os
import glob
import cv2
from PIL import Image as PILImage
from tqdm import tqdm
import shutil

ZIP_CACHE = '/content/drive/MyDrive/fire_clean.zip'
TARGET_DIR = '/content/dataset_clean'

if os.path.exists(ZIP_CACHE):
    print("Đã tìm thấy fire_clean.zip trên Google Drive!")
    print("Đang giải nén dữ liệu sạch trong vài giây...")
    !unzip -q /content/drive/MyDrive/fire_clean.zip -d /content/
    print("Hoàn tất nạp dữ liệu! Bỏ qua toàn bộ bước tiền xử lý, sẵn sàng train ngay lập tức!")
else:
    print("Chưa có file nén trên Drive. Bắt đầu quy trình tiền xử lý dữ liệu từ raw_fire_dataset...")
    SOURCE_DIR = '/content/drive/MyDrive/raw_fire_dataset'

    # Tạo cấu trúc thư mục sạch gồm cả 3 tập
    for split in ['train', 'val', 'test']:
        os.makedirs(f"{TARGET_DIR}/{split}/images", exist_ok=True)
        os.makedirs(f"{TARGET_DIR}/{split}/labels", exist_ok=True)

    def preprocess_split(split, min_size=100):
        img_dir = os.path.join(SOURCE_DIR, split, 'images')
        lbl_dir = os.path.join(SOURCE_DIR, split, 'labels')
        img_files = glob.glob(f"{img_dir}/*.*")
        print(f"\nĐang tiền xử lý tập [{split.upper()}]: Tổng {len(img_files)} ảnh...")

        valid_cnt, corrupt_cnt = 0, 0
        for img_path in tqdm(img_files):
            try:
                with PILImage.open(img_path) as img:
                    img.verify()
                cv_img = cv2.imread(img_path)
                if cv_img is None:
                    corrupt_cnt += 1
                    continue
                h, w = cv_img.shape[:2]
                if h < min_size or w < min_size:
                    corrupt_cnt += 1
                    continue
            except Exception:
                corrupt_cnt += 1
                continue

            base_name = os.path.splitext(os.path.basename(img_path))[0]
            ext = os.path.splitext(img_path)[1].lower()
            if ext not in ['.jpg', '.jpeg', '.png']:
                ext = '.jpg'

            target_img = f"{TARGET_DIR}/{split}/images/{base_name}{ext}"
            target_lbl = f"{TARGET_DIR}/{split}/labels/{base_name}.txt"
            cv2.imwrite(target_img, cv_img)

            src_lbl = os.path.join(lbl_dir, f"{base_name}.txt")
            cleaned_lines = []
            if os.path.exists(src_lbl):
                with open(src_lbl, 'r', encoding='utf-8') as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) >= 5:
                            cls_id = 0
                            try:
                                x, y, bw, bh = map(float, parts[1:5])
                                x = max(0.0, min(1.0, x))
                                y = max(0.0, min(1.0, y))
                                bw = max(0.0, min(1.0, bw))
                                bh = max(0.0, min(1.0, bh))
                                if bw > 0.001 and bh > 0.001:
                                    cleaned_lines.append(f"{cls_id} {x:.6f} {y:.6f} {bw:.6f} {bh:.6f}\n")
                            except ValueError:
                                continue
            with open(target_lbl, 'w', encoding='utf-8') as f:
                f.writelines(cleaned_lines)
            valid_cnt += 1
        print(f"Hoàn tất [{split}]: {valid_cnt} ảnh hợp lệ (loại {corrupt_cnt} ảnh lỗi/không đạt chuẩn).")

    # Tiền xử lý trực tiếp cả 3 tập đã có sẵn
    for split in ['train', 'val', 'test']:
        preprocess_split(split)

    print("\nĐang nén và lưu fire_clean.zip vào Google Drive...")
    !zip -r -q /content/drive/MyDrive/fire_clean.zip /content/dataset_clean
    print("Đã lưu fire_clean.zip lên Google Drive thành công! Các lần sau bạn chỉ mất 5s nạp dữ liệu!")

Chưa có file nén trên Drive. Bắt đầu quy trình tiền xử lý dữ liệu từ raw_fire_dataset...

Đang tiền xử lý tập [TRAIN]: Tổng 2773 ảnh...


100%|██████████| 2773/2773 [18:22<00:00,  2.52it/s]


Hoàn tất [train]: 2773 ảnh hợp lệ (loại 0 ảnh lỗi/không đạt chuẩn).

Đang tiền xử lý tập [VAL]: Tổng 601 ảnh...


100%|██████████| 601/601 [03:40<00:00,  2.72it/s]


Hoàn tất [val]: 601 ảnh hợp lệ (loại 0 ảnh lỗi/không đạt chuẩn).

Đang tiền xử lý tập [TEST]: Tổng 603 ảnh...


100%|██████████| 603/603 [03:40<00:00,  2.73it/s]


Hoàn tất [test]: 603 ảnh hợp lệ (loại 0 ảnh lỗi/không đạt chuẩn).

Đang nén và lưu fire_clean.zip vào Google Drive...
Đã lưu fire_clean.zip lên Google Drive thành công! Các lần sau bạn chỉ mất 5s nạp dữ liệu!


### Chia bộ dữ liệu 70/15/15

In [ ]:
import os

ROOT_DIR = '/content/dataset_clean'

# Chỉ hiển thị thống kê dữ liệu sau khi làm sạch từ raw_fire_dataset
print("THỐNG KÊ BỘ DỮ LIỆU ĐÃ LÀM SẠCH (Từ raw_fire_dataset gốc):")
for split in ['train', 'val', 'test']:
    img_path = f"{ROOT_DIR}/{split}/images"
    if os.path.exists(img_path):
        print(f"Tập {split.upper()}: {len(os.listdir(img_path))} ảnh")
    else:
        print(f"Tập {split.upper()}: Chưa có hoặc không tìm thấy thư mục ảnh")

THỐNG KÊ BỘ DỮ LIỆU ĐÃ LÀM SẠCH (Từ raw_fire_dataset gốc):
Tập TRAIN: 2773 ảnh
Tập VAL: 601 ảnh
Tập TEST: 603 ảnh

Đã tạo cấu hình file: /content/yolov5/fire.yaml
